In [1]:
from dotenv import load_dotenv
from os import getenv
from utils import CensusDataLoader
import pandas as pd
import numpy as np
import os

import glob
from pygris import tracts

load_dotenv()


/mnt/ssd/Projects/RHIP/utils/acs_loader.py:4: UserWarning: Mapping functions unavailable due to import error: ModuleNotFoundError. To use mapping features, ensure all dependencies are properly installed: pip install pytidycensus[map]
  import pytidycensus as tc


True

# Collect ACS Data

In [2]:
acs_loader = CensusDataLoader(api_key=getenv("CENSUS_API_KEY"))

Census API key has been set for this session.


In [ ]:
MOBILITY_VARS = {"MOBILITY_TOTAL": "B07012_002E", "SAME_HOUSE": "B07012_006E", "MOVED_SAME_COUNTY": "B07012_010E", "MOVED_DIFF_COUNTY": "B07012_014E", "MOVED_DIFF_STATE": "B07012_018E", "MOVED_ABROAD": "B07012_022E"}
SNAP_VARS = {"SNAP_TOTAL": "B22003_001E", "RECEIVING": "B22003_002E"}
EMPLOYMENT_VARS = {"LABOR_FORCE": "B23025_002E", "UNEMPLOYED": "B23025_005E"}
EDUCATION_VARS = {"EDU_TOTAL": "B15003_001E", "GRADES": [f"B15003_{num:03d}E" for num in range(2, 17)]}
QUALITY_VARS = {"PLUMB_TOTAL": "B25047_001E", "LACK_PLUMBING": "B25047_003E", "KITCH_TOTAL": "B25051_001E", "LACK_KITCHEN": "B25051_003E"}
POVERTY_VARS = {"POV_TOTAL": "B17001_001E", "POVERTY": "B17001_002E"}
MORTGAGE_VARS = {"MORT_TOTAL": "B25091_001E", "MORT_30_35": "B25091_008E", "MORT_35_40": "B25091_009E", "MORT_40_50": "B25091_010E", "MORT_50_PLUS": "B25091_011E", "NO_MORT_30_35": "B25091_019E", "NO_MORT_35_40": "B25091_020E", "NO_MORT_40_50": "B25091_021E", "NO_MORT_50_PLUS": "B25091_022E"}
RENT_VARS = {"RENT_TOTAL": "B25070_001E", "RENT_30_35": "B25070_007E", "RENT_35_40": "B25070_008E", "RENT_40_50": "B25070_009E", "RENT_50_PLUS": "B25070_010E"}
MISC_VARS = {"GINI": "B19083_001E"}

all_vars_dict = {}
for d in [MOBILITY_VARS, SNAP_VARS, EMPLOYMENT_VARS, EDUCATION_VARS, QUALITY_VARS, POVERTY_VARS, MORTGAGE_VARS, RENT_VARS, MISC_VARS]:
    for key, val in d.items():
        if isinstance(val, list):
            for i, item in enumerate(val): all_vars_dict[f"{key}_{i}"] = item
        else: all_vars_dict[key] = val

In [23]:
# Combine all variables into a single dictionary
all_vars_dict = {}

for d in [
    MOBILITY_VARS,
    SNAP_VARS,
    EMPLOYMENT_VARS,
    EDUCATION_VARS,
    QUALITY_VARS,
    POVERTY_VARS,
    MORTGAGE_VARS,
    RENT_VARS,
]:
    for key, val in d.items():
        if isinstance(val, list):
            for i, item in enumerate(val):
                all_vars_dict[f"{key}_{i}"] = item
        else:
            all_vars_dict[key] = val

In [24]:
def safe_div(num, den):
    return np.where(den == 0, np.nan, num / den)


if "acs_data" not in locals():
    print("Processing ACS Variables")
    acs_data = acs_loader.fetch_multiple(variables=all_vars_dict)
    
    # Rates
    mobility_cols = ["MOVED_SAME_COUNTY", "MOVED_DIFF_COUNTY", "MOVED_DIFF_STATE", "MOVED_ABROAD"]
    acs_data["RiskyMobility"] = safe_div(acs_data[mobility_cols].sum(axis=1), acs_data["MOBILITY_TOTAL"])
    acs_data["SnapRate"] = safe_div(acs_data["RECEIVING"], acs_data["SNAP_TOTAL"])
    acs_data["Unemployment"] = safe_div(acs_data["UNEMPLOYED"], acs_data["LABOR_FORCE"])
    edu_grades = [f"GRADES_{i}" for i in range(15)]
    acs_data["No_HS"] = safe_div(acs_data[edu_grades].sum(axis=1), acs_data["EDU_TOTAL"])
    acs_data["HousingQuality"] = safe_div(acs_data["LACK_PLUMBING"] + acs_data["LACK_KITCHEN"], acs_data["PLUMB_TOTAL"])
    acs_data["Poverty"] = safe_div(acs_data["POVERTY"], acs_data["POV_TOTAL"])

    mort_cols = ["MORT_30_35", "MORT_35_40", "MORT_40_50", "MORT_50_PLUS", "NO_MORT_30_35", "NO_MORT_35_40", "NO_MORT_40_50", "NO_MORT_50_PLUS"]
    rent_cols = ["RENT_30_35", "RENT_35_40", "RENT_40_50", "RENT_50_PLUS"]
    acs_data["MortgageRisk"] = acs_data[mort_cols].sum(axis=1)
    acs_data["RentRisk"] = acs_data[rent_cols].sum(axis=1)
    acs_data["CostBurden"] = safe_div(acs_data["MortgageRisk"] + acs_data["RentRisk"], acs_data["RENT_TOTAL"] + acs_data["MORT_TOTAL"])
        
    # Cleanup
    raw_cols = [c for c in all_vars_dict.keys() if c != "GINI"]
    raw_cols.extend(['MortgageRisk', 'RentRisk'])
    acs_data = acs_data.drop(columns=raw_cols, errors="ignore")
    # acs_data.to_csv("processed_acs_data.csv", index=False)

else: 
    print("ACS Variables already processed")

acs_data.head()

ACS Variables already processed


,GEOID,state,county,tract,NAME,RiskyMobility,SnapRate,Unemployment,No_HS,HousingQuality,Poverty,CostBurden
0,11001000101,11,001,000101,"District of Columbia, DC",0.000000,0.000000,0.023720,0.017195,0.000000,0.017433,0.305523
1,11001000102,11,001,000102,"District of Columbia, DC",0.100671,0.040650,0.012959,0.029379,0.114067,0.049983,0.257453
2,11001000201,11,001,000201,"District of Columbia, DC",0.000000,NaN,0.090528,0.000000,NaN,1.000000,NaN
3,11001000202,11,001,000202,"District of Columbia, DC",0.636569,0.000000,0.027879,0.010775,0.000000,0.136560,0.315963
4,11001000300,11,001,000300,"District of Columbia, DC",0.430605,0.013944,0.008787,0.022262,0.029787,0.046927,0.236255


In [ ]:
REPO_PATH = os.getenv("VIRA_REPO")


In [ ]:
if "ice_data" not in locals():

    ice_dir = f"{REPO_PATH}/facilities/by_state"

    file_pattern = os.path.join(ice_dir, "*.csv")
    csvs = glob.glob(file_pattern)
    ice_data = pd.concat([pd.read_csv(csv) for csv in csvs], ignore_index=True, sort=False)
    ice_data["date"] = pd.to_datetime(ice_data["date"])
    ice_data["year"] = ice_data["date"].dt.year
    ice_data["month"] = ice_data["date"].dt.month
    ice_data = ice_data[ice_data["year"].isin([2013, 2018, 2023])]

ice_data.head()


,detention_facility_code,detention_facility_name,state,date,daily_pop,midnight_pop,year,month
1553,ADACOID,Ada County Jail,ID,2013-01-01,0.0,0.0,2013,1
1554,ADACOID,Ada County Jail,ID,2013-01-02,0.0,0.0,2013,1
1555,ADACOID,Ada County Jail,ID,2013-01-03,0.0,0.0,2013,1
1556,ADACOID,Ada County Jail,ID,2013-01-04,0.0,0.0,2013,1
1557,ADACOID,Ada County Jail,ID,2013-01-05,0.0,0.0,2013,1


In [ ]:
if "facilities" not in locals():
    facilities = pd.read_csv(f"{REPO_PATH}/metadata/facilities.csv")
    facilities.drop(columns=["address", "city", "zip"], inplace=True)

    facilities.rename(columns={"FIPS": "GEOID"}, inplace=True)
    facilities.drop(columns=["detention_facility_name", "state"], inplace=True)
    facilities.sort_values("county", inplace=True)

    facilities.head()

,detention_facility_code,county,aor,latitude,longitude,type_detailed,type_grouped
130,BOIHOLD,Ada,Salt Lake City,43.594251,-116.287523,Hold,Hold/Staging
6,ADACOID,Ada,Salt Lake City,43.607245,-116.269910,IGSA,Non-Dedicated
7,ADAIRKY,Adair,Chicago,37.103817,-85.306621,Other,Other/Unknown
8,ADAMSCO,Adams,Denver,39.991290,-104.798297,IGSA,Non-Dedicated
117,BIINCCO,Adams,Denver,39.760890,-104.849106,Unknown,Other/Unknown


In [ ]:
detention_data = pd.merge(
    facilities,
    ice_data,
    on="detention_facility_code",
    how="left",
)

detention_data.head()

,detention_facility_code,county,aor,latitude,longitude,type_detailed,type_grouped,detention_facility_name,state,date,daily_pop,midnight_pop,year,month
0,BOIHOLD,Ada,Salt Lake City,43.594251,-116.287523,Hold,Hold/Staging,Boise Hold Room,ID,2013-01-01,0.0,0.0,2013.0,1.0
1,BOIHOLD,Ada,Salt Lake City,43.594251,-116.287523,Hold,Hold/Staging,Boise Hold Room,ID,2013-01-02,0.0,0.0,2013.0,1.0
2,BOIHOLD,Ada,Salt Lake City,43.594251,-116.287523,Hold,Hold/Staging,Boise Hold Room,ID,2013-01-03,0.0,0.0,2013.0,1.0
3,BOIHOLD,Ada,Salt Lake City,43.594251,-116.287523,Hold,Hold/Staging,Boise Hold Room,ID,2013-01-04,0.0,0.0,2013.0,1.0
4,BOIHOLD,Ada,Salt Lake City,43.594251,-116.287523,Hold,Hold/Staging,Boise Hold Room,ID,2013-01-05,0.0,0.0,2013.0,1.0


['Alabama',
 'Arizona',
 'Arkansas',
 'California',
 'Colorado',
 'Connecticut',
 'Delaware',
 'District of Columbia',
 'Florida',
 'Georgia',
 'Idaho',
 'Illinois',
 'Indiana',
 'Iowa',
 'Kansas',
 'Kentucky',
 'Louisiana',
 'Maine',
 'Maryland',
 'Massachusetts',
 'Michigan',
 'Minnesota',
 'Mississippi',
 'Missouri',
 'Montana',
 'Nebraska',
 'Nevada',
 'New Hampshire',
 'New Jersey',
 'New Mexico',
 'New York',
 'North Carolina',
 'North Dakota',
 'Ohio',
 'Oklahoma',
 'Oregon',
 'Pennsylvania',
 'Rhode Island',
 'South Carolina',
 'South Dakota',
 'Tennessee',
 'Texas',
 'Utah',
 'Vermont',
 'Virginia',
 'Washington',
 'West Virginia',
 'Wisconsin',
 'Wyoming']

In [ ]:
import sys
from contextlib import redirect_stdout

states_to_remove = [
    "PR",
    "HI",
    "GU",
    "MP",
    "AS",
    "AK",
    "VI",
]  # Filter states to continental US + D.C.


        


In [37]:
tracts

,STATEFP,COUNTYFP,TRACTCE,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,01,063,060200,1400000US01063060200,01063060200,602,Census Tract 602,AL,Greene County,Alabama,CT,570825018,17802697,"POLYGON ((-88.11676 32.69853, -88.11586 32.701..."
1,01,089,010623,1400000US01089010623,01089010623,106.23,Census Tract 106.23,AL,Madison County,Alabama,CT,12933482,45356,"POLYGON ((-86.74938 34.77388, -86.74924 34.779..."
2,01,089,002200,1400000US01089002200,01089002200,22,Census Tract 22,AL,Madison County,Alabama,CT,1575713,0,"POLYGON ((-86.64412 34.71619, -86.62778 34.720..."
3,01,073,000400,1400000US01073000400,01073000400,4,Census Tract 4,AL,Jefferson County,Alabama,CT,7998768,0,"POLYGON ((-86.78617 33.5658, -86.77908 33.5715..."
4,01,073,003300,1400000US01073003300,01073003300,33,Census Tract 33,AL,Jefferson County,Alabama,CT,1973655,0,"POLYGON ((-86.90209 33.51725, -86.89969 33.519..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83501,56,045,951100,1400000US56045951100,56045951100,9511,Census Tract 9511,WY,Weston County,Wyoming,CT,6100000350,5041727,"POLYGON ((-105.08078 43.96622, -105.07928 44.1..."
83502,56,039,967701,1400000US56039967701,56039967701,9677.01,Census Tract 9677.01,WY,Teton County,Wyoming,CT,60998666,93329,"POLYGON ((-110.77132 43.47762, -110.76925 43.4..."
83503,56,005,000500,1400000US56005000500,56005000500,5,Census Tract 5,WY,Campbell County,Wyoming,CT,35210188,16162,"POLYGON ((-105.6605 44.2448, -105.64422 44.248..."
83504,56,041,975202,1400000US56041975202,56041975202,9752.02,Census Tract 9752.02,WY,Uinta County,Wyoming,CT,2904854626,4462874,"POLYGON ((-110.74272 41.29317, -110.7421 41.29..."
